In [ ]:
# ============================================================
# FIXED KAGGLE ONE-CELL STARTER
# Correctly uses BOTH files:
# 1) raw-domain file: Domain + Label, 2,000,000 rows
# 2) processed feature file: 23 features + Label, 1,048,575 rows
# ============================================================

import os, gc, time, random, shutil, warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    brier_score_loss, log_loss, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

try:
    from scipy.stats import ks_2samp, wasserstein_distance
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except Exception as e:
    XGB_OK = False
    raise RuntimeError("XGBoost is required in this Kaggle notebook.") from e

try:
    import shap
    SHAP_OK = True
except Exception:
    SHAP_OK = False

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

ROUND_PRECISION = 4
LOW_FPR_LEVELS = [0.01, 0.005, 0.001]
QUICK_MODE = False
QUICK_SAMPLE_ROWS = 150000

OUTPUT_DIR = Path("/kaggle/working/IEEE_Access_Rerun_Results")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "data"

for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("IEEE Access Kaggle rerun started:", datetime.now())
print("Device:", DEVICE)
print("SHAP available:", SHAP_OK)
print("=" * 80)

# ============================================================
# 1. FIND BOTH CSV FILES CORRECTLY
# ============================================================

INPUT_DIR = Path("/kaggle/input")
all_csvs = list(INPUT_DIR.rglob("*.csv"))

print("\nCSV files found:")
for p in all_csvs:
    print(" -", p)

def inspect_csv_columns(path):
    try:
        tmp = pd.read_csv(path, nrows=5)
        return list(tmp.columns), tmp.shape[1]
    except Exception:
        return [], 0

raw_path = None
processed_path = None

for p in all_csvs:
    cols, ncols = inspect_csv_columns(p)
    lower_cols = [c.lower() for c in cols]
    name = p.name.lower()

    # Raw file should have Domain and Label, usually 2 columns
    if ("domain" in lower_cols) and ("label" in lower_cols) and ncols <= 3:
        raw_path = p

    # Processed file should have Label + many numeric feature columns, no Domain column
    if ("label" in lower_cols) and ncols >= 10 and ("domain" not in lower_cols):
        processed_path = p

# Fallback by filename if needed
if raw_path is None:
    for p in all_csvs:
        if "raw" in p.name.lower():
            raw_path = p
            break

if processed_path is None:
    for p in all_csvs:
        if "raw" not in p.name.lower() and "domain_classification_dataset" in p.name.lower():
            cols, ncols = inspect_csv_columns(p)
            if ncols >= 10:
                processed_path = p
                break

if raw_path is None:
    raise FileNotFoundError("Raw-domain dataset not found. Expected file with columns: Domain, Label.")

if processed_path is None:
    raise FileNotFoundError("Processed feature dataset not found. Expected file with 23 features + Label.")

print("\nSelected RAW file:", raw_path)
print("Selected PROCESSED file:", processed_path)

# ============================================================
# 2. LOAD BOTH DATASETS
# ============================================================

raw_df = pd.read_csv(raw_path)
processed_df = pd.read_csv(processed_path)

print("\nRaw-domain dataset loaded successfully.")
print("Raw shape:", raw_df.shape)
display(raw_df.head())

print("\nProcessed feature dataset loaded successfully.")
print("Processed shape:", processed_df.shape)
display(processed_df.head())

# Optional quick mode for testing
if QUICK_MODE and len(processed_df) > QUICK_SAMPLE_ROWS:
    processed_df = processed_df.sample(n=QUICK_SAMPLE_ROWS, random_state=SEED).reset_index(drop=True)
    print("\nQUICK_MODE active. Processed dataset sampled:", processed_df.shape)

# ============================================================
# 3. DETECT COLUMNS
# ============================================================

def detect_label_col(df):
    for c in ["Label", "label", "Class", "class", "target", "Target"]:
        if c in df.columns:
            return c
    raise ValueError("Label column not found.")

def detect_domain_col(df):
    for c in ["Domain", "domain", "URL", "url", "host", "hostname"]:
        if c in df.columns:
            return c
    return None

raw_label_col = detect_label_col(raw_df)
raw_domain_col = detect_domain_col(raw_df)
processed_label_col = detect_label_col(processed_df)

def normalize_labels(s):
    s = s.astype(str).str.strip().str.lower()
    mapping = {
        "benign": 0,
        "normal": 0,
        "legitimate": 0,
        "0": 0,
        "malicious": 1,
        "malware": 1,
        "bad": 1,
        "1": 1
    }
    y = s.map(mapping)
    if y.isna().any():
        print("Unknown labels:", s[y.isna()].unique()[:20])
        raise ValueError("Unknown label values found.")
    return y.astype(int)

raw_df["_y_"] = normalize_labels(raw_df[raw_label_col])
processed_df["_y_"] = normalize_labels(processed_df[processed_label_col])

# IMPORTANT FIX:
# Feature columns must come ONLY from processed_df and exclude Label + helper columns.
feature_cols = [
    c for c in processed_df.columns
    if c not in [processed_label_col, "_y_"]
    and pd.api.types.is_numeric_dtype(processed_df[c])
]

if len(feature_cols) == 0:
    raise ValueError(
        "No numeric feature columns detected in processed dataset. "
        "This means the wrong file is selected as processed_path."
    )

print("\nDetected structure:")
print("Raw domain column:", raw_domain_col)
print("Raw label column:", raw_label_col)
print("Processed label column:", processed_label_col)
print("Processed feature columns:", len(feature_cols))
print(feature_cols)

# ============================================================
# 4. DATASET STATISTICS FOR IEEE TABLE
# ============================================================

def get_class_stats(df):
    vc = df["_y_"].value_counts().sort_index()
    total = len(df)
    benign = int(vc.get(0, 0))
    malicious = int(vc.get(1, 0))
    return benign, malicious, benign / total * 100, malicious / total * 100

raw_benign, raw_malicious, raw_benign_pct, raw_malicious_pct = get_class_stats(raw_df)
proc_benign, proc_malicious, proc_benign_pct, proc_malicious_pct = get_class_stats(processed_df)

raw_duplicates = int(raw_df[[raw_domain_col, raw_label_col]].duplicated().sum())
raw_missing = int(raw_df[[raw_domain_col, raw_label_col]].isna().sum().sum())
raw_unique_domains = int(raw_df[raw_domain_col].nunique())

processed_missing = int(processed_df[feature_cols + [processed_label_col]].isna().sum().sum())
processed_exact_duplicates = int(processed_df[feature_cols + [processed_label_col]].duplicated().sum())

dataset_stats = pd.DataFrame([
    ["Dataset owner/curator", "Authors"],
    ["Collection period", "approximately June 2025 to August 2025"],
    ["Public release", "Kaggle, September 2025"],
    ["Raw dataset file", raw_path.name],
    ["Processed dataset file", processed_path.name],
    ["Raw records", len(raw_df)],
    ["Raw columns", raw_df.shape[1] - 1],
    ["Raw domain column", raw_domain_col],
    ["Raw label column", raw_label_col],
    ["Raw unique domains", raw_unique_domains],
    ["Raw benign records", raw_benign],
    ["Raw malicious records", raw_malicious],
    ["Raw benign %", round(raw_benign_pct, 4)],
    ["Raw malicious %", round(raw_malicious_pct, 4)],
    ["Processed records", len(processed_df)],
    ["Processed columns", processed_df.shape[1] - 1],
    ["Processed feature columns", len(feature_cols)],
    ["Processed benign records", proc_benign],
    ["Processed malicious records", proc_malicious],
    ["Processed benign %", round(proc_benign_pct, 4)],
    ["Processed malicious %", round(proc_malicious_pct, 4)],
    ["Raw exact duplicate rows", raw_duplicates],
    ["Processed exact duplicate feature+label rows", processed_exact_duplicates],
    ["Raw missing cells", raw_missing],
    ["Processed missing cells", processed_missing],
    ["Timestamp field in released CSV", "Not available"],
    ["Verified malware-family column", "Not available"],
    ["Reviewer-safe statement", "No forensic malware-family composition is claimed; rounded-hash proxy groups are used only for leakage control."]
], columns=["Metric", "Value"])

dataset_stats.to_csv(TABLE_DIR / "Table_1_Dataset_Composition.csv", index=False)
display(dataset_stats)

# ============================================================
# 5. PREPARE FEATURE MATRIX FROM PROCESSED FILE ONLY
# ============================================================

X_df = processed_df[feature_cols].copy()
X_df = X_df.replace([np.inf, -np.inf], np.nan).fillna(0)

for c in feature_cols:
    X_df[c] = X_df[c].astype("float32")

y = processed_df["_y_"].values.astype(int)
X = X_df.values.astype("float32")

# ============================================================
# 6. FIXED ROUNDED-HASH PROXY GROUPING
# This avoids the pandas hash bug by hashing row tuples safely.
# ============================================================

print("\nCreating rounded-hash proxy groups from processed feature dataset...")

rounded_np = np.round(X, ROUND_PRECISION)
rounded_df = pd.DataFrame(rounded_np, columns=feature_cols)

# Safer hashing method
proxy_hash = pd.util.hash_pandas_object(rounded_df.astype(str), index=False).to_numpy(dtype="uint64")

if len(proxy_hash) != len(processed_df):
    raise ValueError(f"Hash length mismatch: {len(proxy_hash)} vs {len(processed_df)}")

processed_df["_proxy_group_"] = proxy_hash

n_proxy_groups = int(pd.Series(proxy_hash).nunique())
proxy_group_counts = pd.Series(proxy_hash).value_counts()

leakage_stats = pd.DataFrame([
    ["Rounding precision", ROUND_PRECISION],
    ["Processed records", len(processed_df)],
    ["Feature columns used in hash", len(feature_cols)],
    ["Exact duplicate feature+label rows", processed_exact_duplicates],
    ["Unique rounded-hash proxy groups", n_proxy_groups],
    ["Mean proxy-group size", round(float(proxy_group_counts.mean()), 4)],
    ["Largest proxy-group size", int(proxy_group_counts.max())],
    ["Purpose", "Leakage control only; not malware-family attribution"]
], columns=["Metric", "Value"])

leakage_stats.to_csv(TABLE_DIR / "Table_2_Leakage_Control.csv", index=False)
display(leakage_stats)

del rounded_np, rounded_df
gc.collect()

# ============================================================
# 7. SPLITS: RANDOM, DEDUPLICATED, ROUNDED-HASH GROUP SPLIT
# ============================================================

def make_random_split(X, y):
    idx = np.arange(len(y))
    train_idx, temp_idx = train_test_split(
        idx, test_size=0.40, random_state=SEED, stratify=y
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=SEED, stratify=y[temp_idx]
    )
    return train_idx, val_idx, test_idx

def make_dedup_split(processed_df, X, y):
    dedup_mask = ~processed_df[feature_cols + [processed_label_col]].duplicated().values
    dedup_idx = np.where(dedup_mask)[0]
    y_dedup = y[dedup_idx]
    train_local, val_local, test_local = make_random_split(X[dedup_idx], y_dedup)
    return dedup_idx[train_local], dedup_idx[val_local], dedup_idx[test_local]

def make_group_split(X, y, groups):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.40, random_state=SEED)
    train_idx, temp_idx = next(gss1.split(X, y, groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_rel, test_rel = next(gss2.split(X[temp_idx], y[temp_idx], groups[temp_idx]))

    val_idx = temp_idx[val_rel]
    test_idx = temp_idx[test_rel]
    return train_idx, val_idx, test_idx

protocols = {
    "random_split": make_random_split(X, y),
    "deduplicated_split": make_dedup_split(processed_df, X, y),
    "rounded_hash_group_split": make_group_split(X, y, proxy_hash)
}

split_rows = []
for name, (tr, va, te) in protocols.items():
    train_groups = set(proxy_hash[tr])
    val_groups = set(proxy_hash[va])
    test_groups = set(proxy_hash[te])

    split_rows.append([
        name,
        len(tr), len(va), len(te),
        round(float(y[tr].mean()), 6),
        round(float(y[va].mean()), 6),
        round(float(y[te].mean()), 6),
        len(train_groups),
        len(val_groups),
        len(test_groups),
        len(train_groups.intersection(val_groups)),
        len(train_groups.intersection(test_groups))
    ])

split_table = pd.DataFrame(split_rows, columns=[
    "Protocol", "Train_n", "Val_n", "Test_n",
    "Train_malicious_rate", "Val_malicious_rate", "Test_malicious_rate",
    "Train_proxy_groups", "Val_proxy_groups", "Test_proxy_groups",
    "Train_Val_proxy_group_overlap", "Train_Test_proxy_group_overlap"
])

split_table.to_csv(TABLE_DIR / "Table_3_Evaluation_Protocols.csv", index=False)
display(split_table)

# ============================================================
# 8. DRIFT METRICS
# ============================================================

def psi_score(source, target, bins=10):
    source = np.asarray(source, dtype=float)
    target = np.asarray(target, dtype=float)

    cuts = np.unique(np.quantile(source, np.linspace(0, 1, bins + 1)))
    if len(cuts) <= 2:
        return 0.0

    s_counts, _ = np.histogram(source, bins=cuts)
    t_counts, _ = np.histogram(target, bins=cuts)

    s_pct = s_counts / max(s_counts.sum(), 1)
    t_pct = t_counts / max(t_counts.sum(), 1)

    eps = 1e-6
    return float(np.sum((t_pct - s_pct) * np.log((t_pct + eps) / (s_pct + eps))))

def js_divergence(source, target, bins=30):
    source = np.asarray(source, dtype=float)
    target = np.asarray(target, dtype=float)

    lo = min(source.min(), target.min())
    hi = max(source.max(), target.max())

    if lo == hi:
        return 0.0

    p, edges = np.histogram(source, bins=bins, range=(lo, hi))
    q, _ = np.histogram(target, bins=edges)

    p = p.astype(float) + 1e-12
    q = q.astype(float) + 1e-12
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    return float(0.5 * (np.sum(p * np.log(p / m)) + np.sum(q * np.log(q / m))))

drift_rows = []

for protocol_name, (tr, va, te) in protocols.items():
    for c_i, c in enumerate(feature_cols):
        source = X[tr, c_i]
        target = X[te, c_i]

        if SCIPY_OK:
            ks = float(ks_2samp(source, target).statistic)
            wass = float(wasserstein_distance(source, target))
        else:
            ks = np.nan
            wass = np.nan

        psi = psi_score(source, target)
        js = js_divergence(source, target)

        if psi < 0.10:
            severity = "Low"
        elif psi < 0.25:
            severity = "Moderate"
        else:
            severity = "High"

        drift_rows.append([protocol_name, c, ks, psi, js, wass, severity])

drift_df = pd.DataFrame(drift_rows, columns=[
    "Protocol", "Feature", "KS_statistic", "PSI",
    "JS_divergence", "Wasserstein_distance", "Drift_severity"
])

drift_df.to_csv(TABLE_DIR / "Table_4_Drift_Metrics.csv", index=False)
display(drift_df.groupby("Protocol")[["KS_statistic", "PSI", "JS_divergence", "Wasserstein_distance"]].mean().reset_index())

# ============================================================
# 9. MODEL + CALIBRATION FUNCTIONS
# ============================================================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def expected_calibration_error(y_true, probs, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    probs = np.clip(np.asarray(probs), 1e-7, 1 - 1e-7)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs <= bins[i+1]) if i == 0 else (probs > bins[i]) & (probs <= bins[i+1])
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)

    return float(ece)

def recall_at_fpr(y_true, scores, fpr_level):
    fpr, tpr, thr = roc_curve(y_true, scores)
    valid = np.where(fpr <= fpr_level)[0]
    if len(valid) == 0:
        return 0.0
    return float(np.max(tpr[valid]))

def threshold_at_fpr(y_true, scores, fpr_level):
    fpr, tpr, thr = roc_curve(y_true, scores)
    valid = np.where(fpr <= fpr_level)[0]
    if len(valid) == 0:
        return 1.0
    best = valid[np.argmax(tpr[valid])]
    return float(thr[best])

def full_metrics(y_true, probs, method, protocol):
    probs = np.clip(probs, 1e-7, 1 - 1e-7)

    row = {
        "Protocol": protocol,
        "Method": method,
        "AUROC": roc_auc_score(y_true, probs),
        "AUPRC": average_precision_score(y_true, probs),
        "ECE": expected_calibration_error(y_true, probs),
        "Brier": brier_score_loss(y_true, probs),
        "NLL": log_loss(y_true, probs, labels=[0, 1])
    }

    for fpr in LOW_FPR_LEVELS:
        row[f"Recall@FPR={fpr}"] = recall_at_fpr(y_true, probs, fpr)

    return row

def fit_temperature_scaling(val_probs, y_val):
    val_logits = logit(val_probs)
    best_T = 1.0
    best_loss = np.inf

    for T in np.linspace(0.25, 5.0, 96):
        p = sigmoid(val_logits / T)
        loss = log_loss(y_val, p, labels=[0, 1])
        if loss < best_loss:
            best_loss = loss
            best_T = float(T)

    return best_T

def apply_temperature(probs, T):
    return sigmoid(logit(probs) / T)

def fit_ttt_entropy_temperature(source_probs, target_probs):
    def entropy(p):
        p = np.clip(p, 1e-7, 1 - 1e-7)
        return -p * np.log(p) - (1 - p) * np.log(1 - p)

    source_entropy = entropy(source_probs).mean()
    target_logits = logit(target_probs)

    best_T = 1.0
    best_diff = np.inf

    for T in np.linspace(0.25, 5.0, 96):
        p = sigmoid(target_logits / T)
        diff = abs(entropy(p).mean() - source_entropy)
        if diff < best_diff:
            best_diff = diff
            best_T = float(T)

    return best_T

def qmt_threshold(source_scores, source_threshold, target_scores):
    alert_rate = np.mean(source_scores >= source_threshold)
    alert_rate = min(max(alert_rate, 1e-6), 1 - 1e-6)
    target_tau = np.quantile(target_scores, 1 - alert_rate)
    return float(target_tau), float(alert_rate)

def threshold_metrics(y_true, scores, threshold, method, protocol, fpr_budget):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    return {
        "Protocol": protocol,
        "Method": method,
        "Target_FPR_Budget": fpr_budget,
        "Threshold": threshold,
        "Observed_FPR": fp / max(fp + tn, 1),
        "Recall": tp / max(tp + fn, 1),
        "Precision": tp / max(tp + fp, 1),
        "Alert_Rate": pred.mean(),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn)
    }

def make_xgb_model():
    # Kaggle T4 GPU support depends on installed XGBoost version.
    # This first tries CUDA; if not supported, it falls back safely.
    try:
        model = XGBClassifier(
            n_estimators=400,
            max_depth=7,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
            device="cuda"
        )
        mode = "T4_GPU_cuda"
    except Exception:
        model = XGBClassifier(
            n_estimators=400,
            max_depth=7,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist"
        )
        mode = "CPU_hist"

    return model, mode

# ============================================================
# 10. RUN EXPERIMENTS
# ============================================================

all_metric_rows = []
all_threshold_rows = []
runtime_rows = []
models = {}

for protocol_name, (tr, va, te) in protocols.items():
    print("\n" + "=" * 80)
    print("Running protocol:", protocol_name)
    print("=" * 80)

    X_train, y_train = X[tr], y[tr]
    X_val, y_val = X[va], y[va]
    X_test, y_test = X[te], y[te]

    model, mode = make_xgb_model()

    start = time.time()
    try:
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    except Exception as e:
        print("GPU training failed. Retrying CPU. Error:", str(e)[:200])
        model = XGBClassifier(
            n_estimators=400,
            max_depth=7,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist"
        )
        mode = "CPU_hist_fallback"
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    train_time = time.time() - start
    models[protocol_name] = model

    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    all_metric_rows.append(full_metrics(y_test, test_probs, "XGBoost_uncalibrated", protocol_name))

    # Temperature Scaling
    T_ts = fit_temperature_scaling(val_probs, y_val)
    test_ts = apply_temperature(test_probs, T_ts)
    all_metric_rows.append(full_metrics(y_test, test_ts, f"TemperatureScaling_T={T_ts:.3f}", protocol_name))

    # Platt Scaling
    platt = LogisticRegression(max_iter=1000, random_state=SEED)
    platt.fit(val_probs.reshape(-1, 1), y_val)
    test_platt = platt.predict_proba(test_probs.reshape(-1, 1))[:, 1]
    all_metric_rows.append(full_metrics(y_test, test_platt, "PlattScaling", protocol_name))

    # Isotonic Regression
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(val_probs, y_val)
    test_iso = np.clip(iso.transform(test_probs), 1e-7, 1 - 1e-7)
    all_metric_rows.append(full_metrics(y_test, test_iso, "IsotonicRegression", protocol_name))

    # T-TTT
    T_ttt = fit_ttt_entropy_temperature(val_probs, test_probs)
    test_ttt = apply_temperature(test_probs, T_ttt)
    all_metric_rows.append(full_metrics(y_test, test_ttt, f"T-TTT_T={T_ttt:.3f}", protocol_name))

    # QMT threshold metrics
    for fpr_budget in LOW_FPR_LEVELS:
        source_tau = threshold_at_fpr(y_val, val_probs, fpr_budget)

        all_threshold_rows.append(
            threshold_metrics(y_test, test_probs, source_tau, "FixedSourceThreshold", protocol_name, fpr_budget)
        )

        qmt_tau, alert_rate = qmt_threshold(val_probs, source_tau, test_probs)
        all_threshold_rows.append(
            threshold_metrics(y_test, test_probs, qmt_tau, "QMT", protocol_name, fpr_budget)
        )

        val_ttt = apply_temperature(val_probs, T_ttt)
        source_tau_ttt = threshold_at_fpr(y_val, val_ttt, fpr_budget)
        qmt_tau_ttt, _ = qmt_threshold(val_ttt, source_tau_ttt, test_ttt)
        all_threshold_rows.append(
            threshold_metrics(y_test, test_ttt, qmt_tau_ttt, "QMT_plus_TTT", protocol_name, fpr_budget)
        )

    runtime_rows.append([protocol_name, mode, "training", len(X_train), train_time, train_time / len(X_train)])

    # Reliability diagram
    plt.figure(figsize=(6, 5))
    for probs, label in [
        (test_probs, "Uncalibrated"),
        (test_ts, "Temperature"),
        (test_platt, "Platt"),
        (test_iso, "Isotonic"),
        (test_ttt, "T-TTT")
    ]:
        frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10, strategy="uniform")
        plt.plot(mean_pred, frac_pos, marker="o", label=label)

    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed malicious frequency")
    plt.title(f"Reliability Diagram: {protocol_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"Reliability_Diagram_{protocol_name}.png", dpi=300)
    plt.close()

    gc.collect()

metrics_df = pd.DataFrame(all_metric_rows)
threshold_df = pd.DataFrame(all_threshold_rows)
runtime_df = pd.DataFrame(runtime_rows, columns=[
    "Protocol", "Mode", "Stage", "N", "Runtime_seconds", "Seconds_per_record"
])

metrics_df.to_csv(TABLE_DIR / "Table_5_Calibration_Metrics.csv", index=False)
threshold_df.to_csv(TABLE_DIR / "Table_6_QMT_Operational_Thresholds.csv", index=False)
runtime_df.to_csv(TABLE_DIR / "Table_7_Runtime.csv", index=False)

display(metrics_df)
display(threshold_df)
display(runtime_df)

# ============================================================
# 11. QMT ATTACK-SPIKE STRESS TEST
# ============================================================

print("\nRunning QMT attack-spike stress test...")

stress_rows = []

protocol_name = "rounded_hash_group_split"
tr, va, te = protocols[protocol_name]
model = models[protocol_name]

X_val, y_val = X[va], y[va]
X_test_base, y_test_base = X[te], y[te]

val_probs = model.predict_proba(X_val)[:, 1]
test_probs_base = model.predict_proba(X_test_base)[:, 1]

mal_idx = np.where(y == 1)[0]
ben_idx = np.where(y == 0)[0]

for injection_rate in [0.00, 0.05, 0.10, 0.20, 0.30]:
    n_extra = int(len(te) * injection_rate)

    if n_extra > 0:
        extra_idx = np.random.choice(mal_idx, size=n_extra, replace=True)
        X_stress = np.vstack([X_test_base, X[extra_idx]])
        y_stress = np.concatenate([y_test_base, y[extra_idx]])
    else:
        X_stress = X_test_base
        y_stress = y_test_base

    stress_probs = model.predict_proba(X_stress)[:, 1]

    for fpr_budget in LOW_FPR_LEVELS:
        source_tau = threshold_at_fpr(y_val, val_probs, fpr_budget)

        fixed = threshold_metrics(
            y_stress, stress_probs, source_tau,
            "FixedSourceThreshold",
            f"AttackSpike_{int(injection_rate*100)}pct",
            fpr_budget
        )

        qmt_tau, _ = qmt_threshold(val_probs, source_tau, stress_probs)

        qmt = threshold_metrics(
            y_stress, stress_probs, qmt_tau,
            "QMT",
            f"AttackSpike_{int(injection_rate*100)}pct",
            fpr_budget
        )

        fixed["Injected_Malicious_Rate"] = injection_rate
        qmt["Injected_Malicious_Rate"] = injection_rate

        stress_rows.append(fixed)
        stress_rows.append(qmt)

stress_df = pd.DataFrame(stress_rows)
stress_df.to_csv(TABLE_DIR / "Table_8_QMT_Attack_Spike_Stress_Test.csv", index=False)
display(stress_df)

# ============================================================
# 12. FEATURE IMPORTANCE AND SHAP
# ============================================================

print("\nGenerating feature importance...")

model = models["rounded_hash_group_split"]
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df.to_csv(TABLE_DIR / "Table_9_XGBoost_Feature_Importance.csv", index=False)
display(importance_df.head(20))

plt.figure(figsize=(8, 6))
top = importance_df.head(15).iloc[::-1]
plt.barh(top["Feature"], top["Importance"])
plt.xlabel("Feature importance")
plt.title("Top XGBoost Feature Importances")
plt.tight_layout()
plt.savefig(FIG_DIR / "Figure_Feature_Importance_XGBoost.png", dpi=300)
plt.close()

if SHAP_OK:
    try:
        print("Computing SHAP sample...")
        _, _, te = protocols["rounded_hash_group_split"]
        sample_n = min(10000, len(te))
        shap_idx = np.random.choice(te, size=sample_n, replace=False)
        X_shap = pd.DataFrame(X[shap_idx], columns=feature_cols)

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_shap)

        plt.figure()
        shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "Figure_SHAP_Summary_XGBoost.png", dpi=300, bbox_inches="tight")
        plt.close()

        shap_df = pd.DataFrame({
            "Feature": feature_cols,
            "Mean_abs_SHAP": np.abs(shap_values).mean(axis=0)
        }).sort_values("Mean_abs_SHAP", ascending=False)

        shap_df.to_csv(TABLE_DIR / "Table_10_SHAP_Mean_Importance.csv", index=False)
        display(shap_df.head(20))
    except Exception as e:
        print("SHAP failed, but feature importance completed:", str(e)[:300])

# ============================================================
# 13. SAVE EXCEL AND ZIP
# ============================================================

excel_path = OUTPUT_DIR / "IEEE_Access_Rerun_All_Tables.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    dataset_stats.to_excel(writer, sheet_name="Dataset_Composition", index=False)
    leakage_stats.to_excel(writer, sheet_name="Leakage_Control", index=False)
    split_table.to_excel(writer, sheet_name="Evaluation_Protocols", index=False)
    drift_df.to_excel(writer, sheet_name="Drift_Metrics", index=False)
    metrics_df.to_excel(writer, sheet_name="Calibration_Metrics", index=False)
    threshold_df.to_excel(writer, sheet_name="QMT_Thresholds", index=False)
    runtime_df.to_excel(writer, sheet_name="Runtime", index=False)
    stress_df.to_excel(writer, sheet_name="QMT_Stress_Test", index=False)
    importance_df.to_excel(writer, sheet_name="Feature_Importance", index=False)

# Manuscript-ready note
methodology_note = f"""
IEEE Access revised methodology note

Both dataset files were used.

Raw-domain file:
{raw_path.name}
Rows: {len(raw_df)}
Columns: {raw_df.shape[1] - 1}
Domain column: {raw_domain_col}
Label column: {raw_label_col}
Benign: {raw_benign}
Malicious: {raw_malicious}
Duplicate rows: {raw_duplicates}
Missing cells: {raw_missing}

Processed feature file:
{processed_path.name}
Rows: {len(processed_df)}
Columns including label: {processed_df.shape[1] - 2}
Feature columns: {len(feature_cols)}
Benign: {proc_benign}
Malicious: {proc_malicious}
Exact duplicate feature+label rows: {processed_exact_duplicates}
Missing cells: {processed_missing}

Collection period:
approximately June 2025 to August 2025

Public release:
Kaggle, September 2025

Important reviewer-safe statement:
No verified malware-family column is available in either released CSV file. Therefore, forensic malware-family composition is not claimed. Rounded-hash proxy groups are used only for leakage control and template-aware splitting.
"""

(OUTPUT_DIR / "IEEE_Methodology_Dataset_Note.txt").write_text(methodology_note, encoding="utf-8")

zip_base = "/kaggle/working/IEEE_Access_Rerun_Results"
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)
print("Excel:", excel_path)
print("ZIP:", zip_path)
print("Figures:", FIG_DIR)
print("Tables:", TABLE_DIR)
print("=" * 80)

In [ ]:
# ============================================================
# IEEE ACCESS FINAL REVIEWER-PATCH CELL
# Run AFTER your previous successful rerun cell.
#
# Adds:
# 1. MLP and FT-Transformer baseline results
# 2. Inference latency + memory + QMT/T-TTT overhead table
# 3. Bootstrap confidence intervals + paired bootstrap comparisons
# 4. High-drift protocol with moderate/severe drift metrics
# 5. QMT limitation discussion using attack-spike results
# ============================================================

import os, gc, time, math, shutil, warnings, random
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    brier_score_loss, log_loss, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

try:
    import psutil
    PSUTIL_OK = True
except Exception:
    PSUTIL_OK = False

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    TORCH_OK = False
    DEVICE = "cpu"

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except Exception:
    XGB_OK = False

try:
    from scipy.stats import ks_2samp, wasserstein_distance
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

# ============================================================
# BASIC CHECKS
# ============================================================

required_vars = ["X", "y", "feature_cols", "protocols", "models", "OUTPUT_DIR", "TABLE_DIR", "FIG_DIR"]
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise RuntimeError(
        "This patch cell must be run after the previous full rerun cell. "
        f"Missing variables: {missing_vars}"
    )

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

if TORCH_OK:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path(OUTPUT_DIR)
TABLE_DIR = Path(TABLE_DIR)
FIG_DIR = Path(FIG_DIR)

PATCH_DIR = OUTPUT_DIR / "reviewer_patch"
PATCH_TABLE_DIR = PATCH_DIR / "tables"
PATCH_FIG_DIR = PATCH_DIR / "figures"

for d in [PATCH_DIR, PATCH_TABLE_DIR, PATCH_FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("IEEE Access final reviewer patch started:", datetime.now())
print("Device:", DEVICE)
print("Torch available:", TORCH_OK)
print("XGBoost available:", XGB_OK)
print("Output:", PATCH_DIR)
print("=" * 80)

# ============================================================
# METRIC FUNCTIONS
# ============================================================

LOW_FPR_LEVELS = [0.01, 0.005, 0.001]

def sigmoid_np(z):
    return 1.0 / (1.0 + np.exp(-z))

def logit_np(p):
    p = np.clip(np.asarray(p), 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def expected_calibration_error(y_true, probs, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    probs = np.clip(np.asarray(probs), 1e-7, 1 - 1e-7)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        if i == 0:
            mask = (probs >= bins[i]) & (probs <= bins[i + 1])
        else:
            mask = (probs > bins[i]) & (probs <= bins[i + 1])

        if mask.sum() == 0:
            continue

        acc = y_true[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)

    return float(ece)

def recall_at_fpr(y_true, scores, fpr_level):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    valid = np.where(fpr <= fpr_level)[0]
    if len(valid) == 0:
        return 0.0
    return float(np.max(tpr[valid]))

def threshold_at_fpr(y_true, scores, fpr_level):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    valid = np.where(fpr <= fpr_level)[0]
    if len(valid) == 0:
        return 1.0
    best = valid[np.argmax(tpr[valid])]
    return float(thresholds[best])

def full_metrics(y_true, probs, method, protocol):
    probs = np.clip(np.asarray(probs), 1e-7, 1 - 1e-7)

    row = {
        "Protocol": protocol,
        "Method": method,
        "AUROC": float(roc_auc_score(y_true, probs)),
        "AUPRC": float(average_precision_score(y_true, probs)),
        "ECE": expected_calibration_error(y_true, probs),
        "Brier": float(brier_score_loss(y_true, probs)),
        "NLL": float(log_loss(y_true, probs, labels=[0, 1])),
    }

    for fpr in LOW_FPR_LEVELS:
        row[f"Recall@FPR={fpr}"] = recall_at_fpr(y_true, probs, fpr)

    return row

def fit_temperature_scaling(val_probs, y_val):
    val_logits = logit_np(val_probs)
    best_T = 1.0
    best_loss = np.inf

    for T in np.linspace(0.25, 5.0, 96):
        p = sigmoid_np(val_logits / T)
        loss = log_loss(y_val, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1])
        if loss < best_loss:
            best_loss = loss
            best_T = float(T)

    return best_T

def apply_temperature(probs, T):
    return sigmoid_np(logit_np(probs) / T)

def entropy_np(p):
    p = np.clip(np.asarray(p), 1e-7, 1 - 1e-7)
    return -p * np.log(p) - (1 - p) * np.log(1 - p)

def fit_ttt_entropy_temperature(source_probs, target_probs):
    source_entropy = float(entropy_np(source_probs).mean())
    target_logits = logit_np(target_probs)

    best_T = 1.0
    best_diff = np.inf

    for T in np.linspace(0.25, 5.0, 96):
        p = sigmoid_np(target_logits / T)
        diff = abs(float(entropy_np(p).mean()) - source_entropy)
        if diff < best_diff:
            best_diff = diff
            best_T = float(T)

    return best_T

def qmt_threshold(source_scores, source_threshold, target_scores):
    alert_rate = float(np.mean(source_scores >= source_threshold))
    alert_rate = min(max(alert_rate, 1e-6), 1 - 1e-6)
    tau_target = float(np.quantile(target_scores, 1 - alert_rate))
    return tau_target, alert_rate

def threshold_metrics(y_true, scores, threshold, method, protocol, fpr_budget):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    return {
        "Protocol": protocol,
        "Method": method,
        "Target_FPR_Budget": fpr_budget,
        "Threshold": float(threshold),
        "Observed_FPR": float(fp / max(fp + tn, 1)),
        "Recall": float(tp / max(tp + fn, 1)),
        "Precision": float(tp / max(tp + fp, 1)),
        "Alert_Rate": float(pred.mean()),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
    }

def current_memory_gb():
    if PSUTIL_OK:
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    return np.nan

def current_gpu_memory_gb():
    if TORCH_OK and torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 3)
    return np.nan

# ============================================================
# 1. HIGH-DRIFT PROTOCOL
# ============================================================

print("\nCreating high-drift protocol...")

X_df_patch = pd.DataFrame(X, columns=feature_cols)

# Build a drift score using features likely to separate domain lexical regimes.
candidate_drift_features = [
    "entropy", "digit_ratio", "length", "max_segment_len",
    "std_pos_digit", "mean_pos_digit", "num_digits", "alpha_numeric_ratio"
]

drift_features = [c for c in candidate_drift_features if c in X_df_patch.columns]

if len(drift_features) < 3:
    drift_features = feature_cols[:min(8, len(feature_cols))]

Z = X_df_patch[drift_features].copy()
Z = (Z - Z.mean()) / (Z.std() + 1e-8)
drift_score = Z.mean(axis=1).values

# Use lower/middle distribution as source and upper tail as high-drift target.
low_cut = np.quantile(drift_score, 0.60)
high_cut = np.quantile(drift_score, 0.80)

source_pool = np.where(drift_score <= low_cut)[0]
target_pool = np.where(drift_score >= high_cut)[0]

source_train_idx, source_val_idx = train_test_split(
    source_pool,
    test_size=0.25,
    random_state=SEED,
    stratify=y[source_pool]
)

high_drift_test_idx = target_pool

protocols["high_drift_protocol"] = (source_train_idx, source_val_idx, high_drift_test_idx)

high_drift_summary = pd.DataFrame([
    ["Drift-score features", ", ".join(drift_features)],
    ["Source rule", "drift_score <= 60th percentile"],
    ["Target rule", "drift_score >= 80th percentile"],
    ["Train_n", len(source_train_idx)],
    ["Val_n", len(source_val_idx)],
    ["Test_n", len(high_drift_test_idx)],
    ["Train malicious rate", float(y[source_train_idx].mean())],
    ["Val malicious rate", float(y[source_val_idx].mean())],
    ["Test malicious rate", float(y[high_drift_test_idx].mean())],
], columns=["Metric", "Value"])

high_drift_summary.to_csv(PATCH_TABLE_DIR / "Table_12_High_Drift_Protocol_Definition.csv", index=False)
display(high_drift_summary)

# ============================================================
# 2. HIGH-DRIFT METRICS
# ============================================================

def psi_score(source, target, bins=10):
    source = np.asarray(source, dtype=float)
    target = np.asarray(target, dtype=float)
    cuts = np.unique(np.quantile(source, np.linspace(0, 1, bins + 1)))

    if len(cuts) <= 2:
        return 0.0

    s_counts, _ = np.histogram(source, bins=cuts)
    t_counts, _ = np.histogram(target, bins=cuts)

    s_pct = s_counts / max(s_counts.sum(), 1)
    t_pct = t_counts / max(t_counts.sum(), 1)

    eps = 1e-6
    return float(np.sum((t_pct - s_pct) * np.log((t_pct + eps) / (s_pct + eps))))

def js_divergence(source, target, bins=30):
    source = np.asarray(source, dtype=float)
    target = np.asarray(target, dtype=float)

    lo = min(source.min(), target.min())
    hi = max(source.max(), target.max())

    if lo == hi:
        return 0.0

    p, edges = np.histogram(source, bins=bins, range=(lo, hi))
    q, _ = np.histogram(target, bins=edges)

    p = p.astype(float) + 1e-12
    q = q.astype(float) + 1e-12

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    return float(0.5 * (np.sum(p * np.log(p / m)) + np.sum(q * np.log(q / m))))

print("\nComputing high-drift feature-level metrics...")

high_drift_rows = []
tr, va, te = protocols["high_drift_protocol"]

for j, c in enumerate(feature_cols):
    source = X[tr, j]
    target = X[te, j]

    if SCIPY_OK:
        ks = float(ks_2samp(source, target).statistic)
        wass = float(wasserstein_distance(source, target))
    else:
        ks = np.nan
        wass = np.nan

    psi = psi_score(source, target)
    js = js_divergence(source, target)

    if psi < 0.10:
        severity = "Low"
    elif psi < 0.25:
        severity = "Moderate"
    else:
        severity = "High"

    high_drift_rows.append([
        "high_drift_protocol", c, ks, psi, js, wass, severity
    ])

high_drift_metrics_df = pd.DataFrame(high_drift_rows, columns=[
    "Protocol", "Feature", "KS_statistic", "PSI",
    "JS_divergence", "Wasserstein_distance", "Drift_severity"
])

high_drift_metrics_df.to_csv(PATCH_TABLE_DIR / "Table_13_High_Drift_Metrics.csv", index=False)
display(high_drift_metrics_df.sort_values("PSI", ascending=False).head(15))

high_drift_overall = high_drift_metrics_df[["KS_statistic", "PSI", "JS_divergence", "Wasserstein_distance"]].mean().reset_index()
high_drift_overall.columns = ["Metric", "Mean_Value"]
high_drift_overall.to_csv(PATCH_TABLE_DIR / "Table_14_High_Drift_Overall_Summary.csv", index=False)
display(high_drift_overall)

# ============================================================
# 3. XGBOOST FOR HIGH-DRIFT PROTOCOL
# ============================================================

def make_xgb_model():
    try:
        return XGBClassifier(
            n_estimators=400,
            max_depth=7,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
            device="cuda"
        ), "T4_GPU_cuda"
    except Exception:
        return XGBClassifier(
            n_estimators=400,
            max_depth=7,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist"
        ), "CPU_hist"

print("\nTraining XGBoost on high-drift protocol...")

tr, va, te = protocols["high_drift_protocol"]
X_train_hd, y_train_hd = X[tr], y[tr]
X_val_hd, y_val_hd = X[va], y[va]
X_test_hd, y_test_hd = X[te], y[te]

xgb_hd, xgb_mode_hd = make_xgb_model()

start = time.time()
try:
    xgb_hd.fit(X_train_hd, y_train_hd, eval_set=[(X_val_hd, y_val_hd)], verbose=False)
except Exception as e:
    print("GPU XGBoost failed; retrying CPU. Error:", str(e)[:250])
    xgb_hd = XGBClassifier(
        n_estimators=400,
        max_depth=7,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist"
    )
    xgb_mode_hd = "CPU_hist_fallback"
    xgb_hd.fit(X_train_hd, y_train_hd, eval_set=[(X_val_hd, y_val_hd)], verbose=False)

xgb_hd_train_time = time.time() - start

models["high_drift_protocol"] = xgb_hd

xgb_val_hd = xgb_hd.predict_proba(X_val_hd)[:, 1]
xgb_test_hd = xgb_hd.predict_proba(X_test_hd)[:, 1]

xgb_high_drift_row = full_metrics(y_test_hd, xgb_test_hd, "XGBoost", "high_drift_protocol")
print("XGBoost high-drift metrics:")
display(pd.DataFrame([xgb_high_drift_row]))

# ============================================================
# 4. PYTORCH MLP AND FT-TRANSFORMER BASELINES
# ============================================================

if not TORCH_OK:
    raise RuntimeError("PyTorch is required for MLP and FT-Transformer baselines.")

print("\nTraining MLP and FT-Transformer baselines...")

# Reviewer-efficient protocols: rounded-hash group split + high-drift protocol
DL_PROTOCOLS = ["rounded_hash_group_split", "high_drift_protocol"]

# To avoid GPU memory issues, cap train size for neural baselines.
# This is acceptable if reported as "trained with stratified capped source samples".
DL_MAX_TRAIN = 350000
DL_MAX_VAL = 120000
DL_MAX_TEST = 220000
DL_BATCH_SIZE = 8192
DL_EPOCHS = 8
DL_PATIENCE = 2
LR = 1e-3

def capped_indices(idx, y_all, max_n, seed=SEED):
    idx = np.asarray(idx)
    if len(idx) <= max_n:
        return idx

    # Stratified cap.
    idx0 = idx[y_all[idx] == 0]
    idx1 = idx[y_all[idx] == 1]

    n0 = min(len(idx0), max_n // 2)
    n1 = min(len(idx1), max_n - n0)

    if n1 < max_n // 2 and len(idx0) > n0:
        n0 = min(len(idx0), max_n - n1)

    s0 = np.random.default_rng(seed).choice(idx0, size=n0, replace=False)
    s1 = np.random.default_rng(seed + 1).choice(idx1, size=n1, replace=False)

    out = np.concatenate([s0, s1])
    np.random.default_rng(seed + 2).shuffle(out)
    return out

class MLPBaseline(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

class FTTransformerNumeric(nn.Module):
    """
    Lightweight FT-Transformer-style model for numerical tabular features.
    Each feature becomes one token:
    token_j = value_j * W_j + B_j
    Then transformer encoder processes feature tokens.
    """
    def __init__(self, n_features, d_token=32, n_heads=4, n_layers=2, dropout=0.10):
        super().__init__()
        self.n_features = n_features
        self.d_token = d_token

        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )

    def forward(self, x):
        # x: [batch, features]
        tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)
        cls = self.cls.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.encoder(tokens)
        cls_out = out[:, 0, :]
        return self.head(cls_out).squeeze(1)

def train_torch_model(model, X_train, y_train, X_val, y_val, model_name):
    model = model.to(DEVICE)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )
    val_x = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    val_y = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

    train_loader = DataLoader(
        train_ds,
        batch_size=DL_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    best_val = np.inf
    best_state = None
    no_improve = 0

    start = time.time()

    for epoch in range(DL_EPOCHS):
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            total_loss += loss.item() * xb.size(0)

        model.eval()
        with torch.no_grad():
            val_logits = model(val_x)
            val_loss = loss_fn(val_logits, val_y).item()

        print(f"{model_name} epoch {epoch+1}/{DL_EPOCHS} | train_loss={total_loss/len(train_ds):.5f} | val_loss={val_loss:.5f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= DL_PATIENCE:
            print(f"{model_name}: early stopping at epoch {epoch+1}")
            break

    train_time = time.time() - start

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_time, best_val

def predict_torch_model(model, X_np, batch_size=DL_BATCH_SIZE):
    model.eval()
    preds = []

    ds = TensorDataset(torch.tensor(X_np, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds.append(probs)

    return np.concatenate(preds)

dl_rows = []
dl_runtime_rows = []
dl_hyper_rows = []

dl_models = {}

for protocol_name in DL_PROTOCOLS:
    print("\n" + "=" * 80)
    print("Deep baseline protocol:", protocol_name)
    print("=" * 80)

    tr, va, te = protocols[protocol_name]

    tr_dl = capped_indices(tr, y, DL_MAX_TRAIN, seed=SEED)
    va_dl = capped_indices(va, y, DL_MAX_VAL, seed=SEED + 10)
    te_dl = capped_indices(te, y, DL_MAX_TEST, seed=SEED + 20)

    scaler = StandardScaler()
    X_train_dl = scaler.fit_transform(X[tr_dl]).astype("float32")
    X_val_dl = scaler.transform(X[va_dl]).astype("float32")
    X_test_dl = scaler.transform(X[te_dl]).astype("float32")

    y_train_dl = y[tr_dl].astype("float32")
    y_val_dl = y[va_dl].astype("float32")
    y_test_dl = y[te_dl].astype(int)

    # MLP
    if TORCH_OK and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    mlp = MLPBaseline(in_dim=X_train_dl.shape[1])
    mlp, mlp_train_time, mlp_best_val = train_torch_model(
        mlp, X_train_dl, y_train_dl, X_val_dl, y_val_dl, f"MLP_{protocol_name}"
    )

    start = time.time()
    mlp_probs = predict_torch_model(mlp, X_test_dl)
    mlp_infer_time = time.time() - start

    dl_rows.append(full_metrics(y_test_dl, mlp_probs, "MLP", protocol_name))
    dl_runtime_rows.append([
        protocol_name, "MLP", "training", len(X_train_dl),
        mlp_train_time, mlp_train_time / len(X_train_dl),
        current_memory_gb(), current_gpu_memory_gb()
    ])
    dl_runtime_rows.append([
        protocol_name, "MLP", "inference_test", len(X_test_dl),
        mlp_infer_time, mlp_infer_time / len(X_test_dl),
        current_memory_gb(), current_gpu_memory_gb()
    ])

    dl_models[(protocol_name, "MLP")] = (mlp, scaler)

    # FT-Transformer
    if TORCH_OK and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    ft = FTTransformerNumeric(
        n_features=X_train_dl.shape[1],
        d_token=32,
        n_heads=4,
        n_layers=2,
        dropout=0.10
    )

    ft, ft_train_time, ft_best_val = train_torch_model(
        ft, X_train_dl, y_train_dl, X_val_dl, y_val_dl, f"FTTransformer_{protocol_name}"
    )

    start = time.time()
    ft_probs = predict_torch_model(ft, X_test_dl)
    ft_infer_time = time.time() - start

    dl_rows.append(full_metrics(y_test_dl, ft_probs, "FT-Transformer", protocol_name))
    dl_runtime_rows.append([
        protocol_name, "FT-Transformer", "training", len(X_train_dl),
        ft_train_time, ft_train_time / len(X_train_dl),
        current_memory_gb(), current_gpu_memory_gb()
    ])
    dl_runtime_rows.append([
        protocol_name, "FT-Transformer", "inference_test", len(X_test_dl),
        ft_infer_time, ft_infer_time / len(X_test_dl),
        current_memory_gb(), current_gpu_memory_gb()
    ])

    dl_models[(protocol_name, "FT-Transformer")] = (ft, scaler)

    dl_hyper_rows.extend([
        [protocol_name, "MLP", "Input features", X_train_dl.shape[1]],
        [protocol_name, "MLP", "Hidden layers", "128, 64"],
        [protocol_name, "MLP", "Dropout", 0.20],
        [protocol_name, "MLP", "Batch size", DL_BATCH_SIZE],
        [protocol_name, "MLP", "Optimizer", "AdamW"],
        [protocol_name, "MLP", "Learning rate", LR],
        [protocol_name, "MLP", "Max epochs", DL_EPOCHS],
        [protocol_name, "FT-Transformer", "Input features", X_train_dl.shape[1]],
        [protocol_name, "FT-Transformer", "Token dimension", 32],
        [protocol_name, "FT-Transformer", "Attention heads", 4],
        [protocol_name, "FT-Transformer", "Transformer layers", 2],
        [protocol_name, "FT-Transformer", "Dropout", 0.10],
        [protocol_name, "FT-Transformer", "Batch size", DL_BATCH_SIZE],
        [protocol_name, "FT-Transformer", "Optimizer", "AdamW"],
        [protocol_name, "FT-Transformer", "Learning rate", LR],
        [protocol_name, "FT-Transformer", "Max epochs", DL_EPOCHS],
    ])

    gc.collect()
    if TORCH_OK and torch.cuda.is_available():
        torch.cuda.empty_cache()

dl_metrics_df = pd.DataFrame(dl_rows)
dl_runtime_df = pd.DataFrame(dl_runtime_rows, columns=[
    "Protocol", "Model", "Stage", "N", "Runtime_seconds",
    "Seconds_per_record", "CPU_memory_GB", "GPU_peak_memory_GB"
])
dl_hyper_df = pd.DataFrame(dl_hyper_rows, columns=[
    "Protocol", "Model", "Parameter", "Value"
])

dl_metrics_df.to_csv(PATCH_TABLE_DIR / "Table_15_MLP_FTTransformer_Baseline_Metrics.csv", index=False)
dl_runtime_df.to_csv(PATCH_TABLE_DIR / "Table_16_MLP_FTTransformer_Runtime.csv", index=False)
dl_hyper_df.to_csv(PATCH_TABLE_DIR / "Table_17_MLP_FTTransformer_Hyperparameters.csv", index=False)

print("\nMLP and FT-Transformer metrics:")
display(dl_metrics_df)

print("\nMLP and FT-Transformer hyperparameters:")
display(dl_hyper_df)

# ============================================================
# 5. XGBOOST + CALIBRATION + QMT ON HIGH-DRIFT PROTOCOL
# ============================================================

print("\nComputing XGBoost calibration and QMT/T-TTT for high-drift protocol...")

high_drift_results = []

T_ts_hd = fit_temperature_scaling(xgb_val_hd, y_val_hd)
xgb_test_hd_ts = apply_temperature(xgb_test_hd, T_ts_hd)

platt_hd = LogisticRegression(max_iter=1000, random_state=SEED)
platt_hd.fit(xgb_val_hd.reshape(-1, 1), y_val_hd)
xgb_test_hd_platt = platt_hd.predict_proba(xgb_test_hd.reshape(-1, 1))[:, 1]

iso_hd = IsotonicRegression(out_of_bounds="clip")
iso_hd.fit(xgb_val_hd, y_val_hd)
xgb_test_hd_iso = np.clip(iso_hd.transform(xgb_test_hd), 1e-7, 1 - 1e-7)

T_ttt_hd = fit_ttt_entropy_temperature(xgb_val_hd, xgb_test_hd)
xgb_test_hd_ttt = apply_temperature(xgb_test_hd, T_ttt_hd)

high_drift_results.append(full_metrics(y_test_hd, xgb_test_hd, "XGBoost_uncalibrated", "high_drift_protocol"))
high_drift_results.append(full_metrics(y_test_hd, xgb_test_hd_ts, f"TemperatureScaling_T={T_ts_hd:.3f}", "high_drift_protocol"))
high_drift_results.append(full_metrics(y_test_hd, xgb_test_hd_platt, "PlattScaling", "high_drift_protocol"))
high_drift_results.append(full_metrics(y_test_hd, xgb_test_hd_iso, "IsotonicRegression", "high_drift_protocol"))
high_drift_results.append(full_metrics(y_test_hd, xgb_test_hd_ttt, f"T-TTT_T={T_ttt_hd:.3f}", "high_drift_protocol"))

high_drift_cal_df = pd.DataFrame(high_drift_results)
high_drift_cal_df.to_csv(PATCH_TABLE_DIR / "Table_18_High_Drift_XGBoost_Calibration_Metrics.csv", index=False)
display(high_drift_cal_df)

high_drift_qmt_rows = []

for fpr_budget in LOW_FPR_LEVELS:
    source_tau = threshold_at_fpr(y_val_hd, xgb_val_hd, fpr_budget)

    fixed = threshold_metrics(
        y_test_hd, xgb_test_hd, source_tau,
        "FixedSourceThreshold", "high_drift_protocol", fpr_budget
    )

    qmt_tau, alert_rate = qmt_threshold(xgb_val_hd, source_tau, xgb_test_hd)

    qmt = threshold_metrics(
        y_test_hd, xgb_test_hd, qmt_tau,
        "QMT", "high_drift_protocol", fpr_budget
    )

    val_ttt_hd = apply_temperature(xgb_val_hd, T_ttt_hd)
    source_tau_ttt = threshold_at_fpr(y_val_hd, val_ttt_hd, fpr_budget)
    qmt_tau_ttt, _ = qmt_threshold(val_ttt_hd, source_tau_ttt, xgb_test_hd_ttt)

    qmt_ttt = threshold_metrics(
        y_test_hd, xgb_test_hd_ttt, qmt_tau_ttt,
        "QMT_plus_TTT", "high_drift_protocol", fpr_budget
    )

    high_drift_qmt_rows.extend([fixed, qmt, qmt_ttt])

high_drift_qmt_df = pd.DataFrame(high_drift_qmt_rows)
high_drift_qmt_df.to_csv(PATCH_TABLE_DIR / "Table_19_High_Drift_QMT_Operational_Metrics.csv", index=False)
display(high_drift_qmt_df)

# ============================================================
# 6. INFERENCE LATENCY + MEMORY + QMT/T-TTT OVERHEAD TABLE
# ============================================================

print("\nMeasuring inference latency, memory, QMT overhead, and T-TTT overhead...")

latency_rows = []
BATCHES = [10000, 100000, 1000000]

def sample_batch(X_source, n):
    if len(X_source) >= n:
        return X_source[:n]
    idx = np.random.default_rng(SEED).choice(np.arange(len(X_source)), size=n, replace=True)
    return X_source[idx]

# Use rounded-hash group split as primary deployment protocol.
lat_protocol = "rounded_hash_group_split"
tr, va, te = protocols[lat_protocol]
xgb_model = models[lat_protocol]
X_val_lat = X[va]
y_val_lat = y[va]
X_test_lat = X[te]

val_scores_lat = xgb_model.predict_proba(X_val_lat)[:, 1]

for n in BATCHES:
    X_batch = sample_batch(X_test_lat, n)

    # XGBoost inference
    start = time.time()
    batch_scores = xgb_model.predict_proba(X_batch)[:, 1]
    infer_time = time.time() - start

    latency_rows.append([
        lat_protocol, "XGBoost", "Inference", n,
        infer_time, infer_time / n,
        current_memory_gb(), current_gpu_memory_gb()
    ])

    # QMT overhead only
    source_tau = threshold_at_fpr(y_val_lat, val_scores_lat, 0.001)
    start = time.time()
    tau_qmt, alert_rate_qmt = qmt_threshold(val_scores_lat, source_tau, batch_scores)
    qmt_time = time.time() - start

    latency_rows.append([
        lat_protocol, "QMT", "Threshold_update_only", n,
        qmt_time, qmt_time / n,
        current_memory_gb(), current_gpu_memory_gb()
    ])

    # T-TTT overhead only
    start = time.time()
    T_ttt_lat = fit_ttt_entropy_temperature(val_scores_lat, batch_scores)
    batch_ttt = apply_temperature(batch_scores, T_ttt_lat)
    ttt_time = time.time() - start

    latency_rows.append([
        lat_protocol, "T-TTT", "Temperature_update_only", n,
        ttt_time, ttt_time / n,
        current_memory_gb(), current_gpu_memory_gb()
    ])

    # QMT + T-TTT overhead
    start = time.time()
    val_ttt_lat = apply_temperature(val_scores_lat, T_ttt_lat)
    source_tau_ttt = threshold_at_fpr(y_val_lat, val_ttt_lat, 0.001)
    tau_qmt_ttt, alert_rate_ttt = qmt_threshold(val_ttt_lat, source_tau_ttt, batch_ttt)
    combo_time = time.time() - start

    latency_rows.append([
        lat_protocol, "QMT_plus_TTT", "Combined_update_only", n,
        combo_time, combo_time / n,
        current_memory_gb(), current_gpu_memory_gb()
    ])

latency_df = pd.DataFrame(latency_rows, columns=[
    "Protocol", "Method", "Stage", "Batch_size",
    "Runtime_seconds", "Seconds_per_record",
    "CPU_memory_GB", "GPU_peak_memory_GB"
])

latency_df.to_csv(PATCH_TABLE_DIR / "Table_20_Inference_Latency_Memory_QMT_TTT_Overhead.csv", index=False)
display(latency_df)

# ============================================================
# 7. BOOTSTRAP CONFIDENCE INTERVALS AND PAIRED BOOTSTRAP
# ============================================================

print("\nComputing bootstrap confidence intervals and paired comparisons...")

BOOT_B = 300
BOOT_MAX_N = 80000

def bootstrap_ci(y_true, probs, metric_fn, B=BOOT_B, max_n=BOOT_MAX_N, seed=SEED):
    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs)

    n = len(y_true)
    if n > max_n:
        base_idx = rng.choice(np.arange(n), size=max_n, replace=False)
        y_true = y_true[base_idx]
        probs = probs[base_idx]
        n = len(y_true)

    vals = []

    for _ in range(B):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            vals.append(metric_fn(y_true[idx], probs[idx]))
        except Exception:
            pass

    vals = np.asarray(vals)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan

    return float(vals.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def paired_bootstrap_delta(y_true, probs_a, probs_b, metric_fn, B=BOOT_B, max_n=BOOT_MAX_N, seed=SEED):
    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true).astype(int)
    probs_a = np.asarray(probs_a)
    probs_b = np.asarray(probs_b)

    n = len(y_true)
    if n > max_n:
        base_idx = rng.choice(np.arange(n), size=max_n, replace=False)
        y_true = y_true[base_idx]
        probs_a = probs_a[base_idx]
        probs_b = probs_b[base_idx]
        n = len(y_true)

    deltas = []

    for _ in range(B):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            deltas.append(metric_fn(y_true[idx], probs_b[idx]) - metric_fn(y_true[idx], probs_a[idx]))
        except Exception:
            pass

    deltas = np.asarray(deltas)
    if len(deltas) == 0:
        return np.nan, np.nan, np.nan, np.nan

    p_two = float(2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0)))
    p_two = min(max(p_two, 0.0), 1.0)

    return float(deltas.mean()), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)), p_two

metric_fns = {
    "AUROC": lambda yy, pp: roc_auc_score(yy, pp),
    "AUPRC": lambda yy, pp: average_precision_score(yy, pp),
    "ECE": lambda yy, pp: expected_calibration_error(yy, pp),
    "Brier": lambda yy, pp: brier_score_loss(yy, pp),
    "NLL": lambda yy, pp: log_loss(yy, np.clip(pp, 1e-7, 1 - 1e-7), labels=[0, 1]),
    "Recall@FPR=0.001": lambda yy, pp: recall_at_fpr(yy, pp, 0.001),
}

bootstrap_rows = []
paired_rows = []

# Use XGBoost high-drift and rounded-hash protocols.
bootstrap_sources = {}

for protocol_name in ["rounded_hash_group_split", "high_drift_protocol"]:
    tr, va, te = protocols[protocol_name]
    model = models[protocol_name]

    y_test_b = y[te]
    probs_xgb = model.predict_proba(X[te])[:, 1]

    bootstrap_sources[(protocol_name, "XGBoost")] = (y_test_b, probs_xgb)

# Add MLP and FT if same protocol outputs need fresh prediction.
for protocol_name in DL_PROTOCOLS:
    tr, va, te = protocols[protocol_name]
    te_dl = capped_indices(te, y, DL_MAX_TEST, seed=SEED + 20)
    y_test_dl = y[te_dl].astype(int)

    for model_name in ["MLP", "FT-Transformer"]:
        if (protocol_name, model_name) not in dl_models:
            continue

        torch_model, scaler = dl_models[(protocol_name, model_name)]
        X_test_dl = scaler.transform(X[te_dl]).astype("float32")
        probs_dl = predict_torch_model(torch_model, X_test_dl)

        bootstrap_sources[(protocol_name, model_name)] = (y_test_dl, probs_dl)

for (protocol_name, method_name), (yy, pp) in bootstrap_sources.items():
    for metric_name, fn in metric_fns.items():
        mean_v, lo, hi = bootstrap_ci(yy, pp, fn)
        bootstrap_rows.append([
            protocol_name, method_name, metric_name,
            mean_v, lo, hi, BOOT_B
        ])

bootstrap_df = pd.DataFrame(bootstrap_rows, columns=[
    "Protocol", "Method", "Metric", "Bootstrap_Mean",
    "CI_2.5", "CI_97.5", "Bootstrap_B"
])

bootstrap_df.to_csv(PATCH_TABLE_DIR / "Table_21_Bootstrap_Confidence_Intervals.csv", index=False)
display(bootstrap_df.head(20))

# Paired model comparison against XGBoost where sample alignment allows.
# For neural capped samples, compare within the same capped test set by predicting XGBoost on the same rows.
for protocol_name in DL_PROTOCOLS:
    tr, va, te = protocols[protocol_name]
    te_dl = capped_indices(te, y, DL_MAX_TEST, seed=SEED + 20)
    yy = y[te_dl].astype(int)
    xgb_probs_aligned = models[protocol_name].predict_proba(X[te_dl])[:, 1]

    for model_name in ["MLP", "FT-Transformer"]:
        if (protocol_name, model_name) not in dl_models:
            continue

        torch_model, scaler = dl_models[(protocol_name, model_name)]
        X_test_dl = scaler.transform(X[te_dl]).astype("float32")
        dl_probs = predict_torch_model(torch_model, X_test_dl)

        for metric_name, fn in metric_fns.items():
            delta_mean, lo, hi, p_two = paired_bootstrap_delta(
                yy, xgb_probs_aligned, dl_probs, fn
            )
            paired_rows.append([
                protocol_name,
                "XGBoost",
                model_name,
                metric_name,
                "Comparator_minus_XGBoost",
                delta_mean,
                lo,
                hi,
                p_two,
                BOOT_B
            ])

paired_df = pd.DataFrame(paired_rows, columns=[
    "Protocol", "Baseline", "Comparator", "Metric",
    "Delta_definition", "Delta_mean", "CI_2.5", "CI_97.5",
    "Approx_two_sided_p", "Bootstrap_B"
])

paired_df.to_csv(PATCH_TABLE_DIR / "Table_22_Paired_Bootstrap_Model_Comparisons.csv", index=False)
display(paired_df.head(20))

# ============================================================
# 8. QMT LIMITATION DISCUSSION FROM ATTACK-SPIKE RESULTS
# ============================================================

print("\nGenerating QMT limitation discussion...")

stress_path_candidates = [
    TABLE_DIR / "Table_8_QMT_Attack_Spike_Stress_Test.csv",
    OUTPUT_DIR / "tables" / "Table_8_QMT_Attack_Spike_Stress_Test.csv"
]

stress_df_loaded = None

for p in stress_path_candidates:
    if Path(p).exists():
        stress_df_loaded = pd.read_csv(p)
        break

if stress_df_loaded is None and "stress_df" in globals():
    stress_df_loaded = stress_df.copy()

if stress_df_loaded is None:
    qmt_limitation_text = """
QMT limitation discussion could not be auto-generated because the attack-spike stress-test table was not found.
Please run the QMT stress-test block first.
"""
else:
    # Focus on FPR budget 0.001 because reviewer cares about ultra-low FPR.
    ss = stress_df_loaded.copy()
    ss = ss[ss["Target_FPR_Budget"].astype(float) == 0.001]

    # Keep only FixedSourceThreshold and QMT.
    ss = ss[ss["Method"].isin(["FixedSourceThreshold", "QMT"])]

    # Create pivot where possible.
    qmt_lines = []

    for proto in sorted(ss["Protocol"].unique()):
        temp = ss[ss["Protocol"] == proto]
        if "AttackSpike" not in str(proto):
            continue

        inj = temp["Injected_Malicious_Rate"].iloc[0] if "Injected_Malicious_Rate" in temp.columns else np.nan

        fixed_recall = temp[temp["Method"] == "FixedSourceThreshold"]["Recall"]
        qmt_recall = temp[temp["Method"] == "QMT"]["Recall"]

        fixed_fpr = temp[temp["Method"] == "FixedSourceThreshold"]["Observed_FPR"]
        qmt_fpr = temp[temp["Method"] == "QMT"]["Observed_FPR"]

        if len(fixed_recall) and len(qmt_recall):
            qmt_lines.append([
                proto,
                inj,
                float(fixed_recall.iloc[0]),
                float(qmt_recall.iloc[0]),
                float(qmt_recall.iloc[0] - fixed_recall.iloc[0]),
                float(fixed_fpr.iloc[0]),
                float(qmt_fpr.iloc[0]),
            ])

    qmt_lim_df = pd.DataFrame(qmt_lines, columns=[
        "Stress_condition", "Injected_malicious_rate",
        "Fixed_threshold_recall", "QMT_recall", "QMT_minus_fixed_recall",
        "Fixed_threshold_observed_FPR", "QMT_observed_FPR"
    ])

    qmt_lim_df.to_csv(PATCH_TABLE_DIR / "Table_23_QMT_AttackSpike_Limitation_Summary.csv", index=False)
    display(qmt_lim_df)

    qmt_limitation_text = f"""
### QMT Failure-Case and Limitation Discussion

The attack-spike stress test was added to examine the main assumption behind Quantile-Matched Thresholding (QMT). QMT is designed to stabilize the operational alert budget by matching the source alert quantile to the target score distribution. This behavior is useful when the main deployment problem is covariate score shift and the security team wants a stable alert volume. However, the stress-test results show that QMT should not be interpreted as a general-purpose attack-surge detector.

At the ultra-low false-positive-rate operating point of FPR = 0.001, QMT preserved a stable alert budget under the no-spike condition. However, when additional malicious samples were injected into the target stream, QMT increased the decision threshold and reduced recall compared with the fixed source threshold. This result confirms the reviewer’s concern that quantile matching can hide true increases in malicious prevalence. In other words, QMT can suppress alert growth during real attack bursts because it treats the shifted target score distribution as a budget-stabilization problem rather than as evidence of a possible increase in malicious activity.

Therefore, the revised manuscript treats QMT as a lightweight alert-budget stabilization method, not as a replacement for prevalence monitoring or incident escalation. In practical SOC deployment, QMT should be combined with independent monitoring signals, including target-score distribution alarms, sudden changes in high-score volume, prevalence-sensitive dashboards, and analyst-defined emergency thresholds. When an attack surge is suspected, QMT should be disabled or used with a guardrail that permits alert-volume expansion.
"""

(PATCH_DIR / "IEEE_QMT_Limitation_Discussion.txt").write_text(qmt_limitation_text, encoding="utf-8")
print(qmt_limitation_text)

# ============================================================
# 9. REVISED RESPONSE-TO-REVIEWER TEXT FOR MISSING ITEMS
# ============================================================

reviewer_patch_response = """
Reviewer-oriented revision summary for the final experimental patch

1. FT-Transformer and MLP baselines:
We added two neural tabular baselines: a multilayer perceptron (MLP) and a lightweight FT-Transformer-style model for numerical tabular data. The revised manuscript reports architecture settings, optimization settings, training details, and performance metrics for both models under the rounded-hash group split and the high-drift protocol.

2. Runtime, memory, and inference overhead:
We added a deployment-cost table reporting inference latency, CPU memory, GPU memory, QMT threshold-update overhead, T-TTT temperature-update overhead, and combined QMT + T-TTT update overhead at batch sizes of 10,000, 100,000, and 1,000,000 domains.

3. Statistical confidence:
We added bootstrap confidence intervals for AUROC, AUPRC, ECE, Brier Score, NLL, and Recall@FPR=0.001. We also added paired bootstrap comparisons between XGBoost and the neural tabular baselines.

4. High-drift protocol:
We added a high-drift evaluation protocol by constructing source and target partitions using a lexical drift score based on entropy, digit ratio, length, maximum segment length, digit-position statistics, and related features. Distribution shift is quantified using KS statistic, Population Stability Index, Jensen-Shannon divergence, and Wasserstein distance.

5. QMT limitation:
We expanded the QMT limitation discussion using the attack-spike stress-test results. The revised text clarifies that QMT stabilizes alert budgets under covariate score shift but may suppress alert growth when malicious prevalence truly increases. Therefore, QMT should be used with prevalence monitoring and SOC guardrails.
"""

(PATCH_DIR / "IEEE_Response_to_Reviewers_Final_Patch_Text.txt").write_text(reviewer_patch_response, encoding="utf-8")

# ============================================================
# 10. SAVE PATCH EXCEL WORKBOOK
# ============================================================

patch_excel_path = PATCH_DIR / "IEEE_Access_Final_Reviewer_Patch_Tables.xlsx"

with pd.ExcelWriter(patch_excel_path, engine="openpyxl") as writer:
    high_drift_summary.to_excel(writer, sheet_name="High_Drift_Protocol", index=False)
    high_drift_metrics_df.to_excel(writer, sheet_name="High_Drift_Metrics", index=False)
    high_drift_overall.to_excel(writer, sheet_name="High_Drift_Overall", index=False)
    dl_metrics_df.to_excel(writer, sheet_name="MLP_FT_Metrics", index=False)
    dl_runtime_df.to_excel(writer, sheet_name="MLP_FT_Runtime", index=False)
    dl_hyper_df.to_excel(writer, sheet_name="MLP_FT_Hyperparameters", index=False)
    high_drift_cal_df.to_excel(writer, sheet_name="High_Drift_Calibration", index=False)
    high_drift_qmt_df.to_excel(writer, sheet_name="High_Drift_QMT", index=False)
    latency_df.to_excel(writer, sheet_name="Latency_Memory_Overhead", index=False)
    bootstrap_df.to_excel(writer, sheet_name="Bootstrap_CI", index=False)
    paired_df.to_excel(writer, sheet_name="Paired_Bootstrap", index=False)

    if "qmt_lim_df" in globals():
        qmt_lim_df.to_excel(writer, sheet_name="QMT_Limitation", index=False)

print("\nPatch Excel saved:", patch_excel_path)

# ============================================================
# 11. ZIP FINAL OUTPUTS
# ============================================================

zip_base = "/kaggle/working/IEEE_Access_Final_Reviewer_Patch"
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(zip_base, "zip", PATCH_DIR)

# Also create full updated ZIP including original rerun + patch.
full_zip_base = "/kaggle/working/IEEE_Access_Rerun_Results_WITH_FINAL_PATCH"
full_zip_path = full_zip_base + ".zip"

if os.path.exists(full_zip_path):
    os.remove(full_zip_path)

shutil.make_archive(full_zip_base, "zip", OUTPUT_DIR)

print("\n" + "=" * 80)
print("FINAL REVIEWER PATCH COMPLETE")
print("=" * 80)
print("Patch folder:", PATCH_DIR)
print("Patch Excel:", patch_excel_path)
print("Patch ZIP:", zip_path)
print("Full updated ZIP:", full_zip_path)
print("\nDownload these from Kaggle output:")
print(zip_path)
print(full_zip_path)
print("=" * 80)

In [ ]:
import os
import shutil
from pathlib import Path

PATCH_DIR = Path("/kaggle/working/IEEE_Access_Rerun_Results/reviewer_patch")

ZIP_BASE = "/kaggle/working/reviewer_patch"
ZIP_PATH = ZIP_BASE + ".zip"

if not PATCH_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {PATCH_DIR}")

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive(ZIP_BASE, "zip", PATCH_DIR)

print("ZIP created successfully:")
print(ZIP_PATH)

print("\nFiles included:")
for p in sorted(PATCH_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(PATCH_DIR))